# Pemodelan dan Evaluasi
Melatih dan membandingkan 3 model klasifikasi status stunting: XGBoost, Random Forest, dan Random Forest dengan GridSearchCV.

## Import Library & Data

In [ ]:
import sys
sys.path.append('../src')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import MinMaxScaler, OneHotEncoder, LabelEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, confusion_matrix, classification_report
from xgboost import XGBClassifier
import warnings
warnings.filterwarnings('ignore')

from utils import metrics_dict

df = pd.read_csv('../data/processed/data_cleaned.csv')
df.head()


## Splitting Data

In [ ]:
print("==== membagi data menjadi training dan testing ===")

# Kolom id, id_anak, jenis_data, tanggal_pengukuran adalah identifier/metadata,
# bukan fitur prediktif -> ikut dibuang bersama target & skor_z_haz/tanggal_lahir
drop_cols = ['status_stunting', 'skor_z_haz', 'tanggal_lahir',
             'id', 'id_anak', 'jenis_data', 'tanggal_pengukuran']
x = df.drop(columns=[c for c in drop_cols if c in df.columns])
y = df['status_stunting']

x_train, x_test, y_train, y_test = train_test_split(x, y, test_size=0.2, random_state=42, stratify=y)

print(f"jumlah data training (x_train): {x_train.shape[0]}")
print(f"jumlah data testing (x_test): {x_test.shape[0]}")


In [ ]:
numeric_features = x_train.select_dtypes(include=np.number).columns.tolist()
categorical_features = x_train.select_dtypes(include=['object', 'category']).columns.tolist()

preprocessor_clf = ColumnTransformer(
    transformers=[
        ('num', MinMaxScaler(), numeric_features),
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
    ],
    remainder='passthrough'
)
print("preprocessor 'preprocessor_clf' telah dibuat.")


## XGBoost

In [ ]:
xgb_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_clf),
    ('classifier', XGBClassifier(
        n_estimators=200,
        learning_rate=0.05,
        max_depth=4,
        subsample=0.8,
        colsample_bytree=0.8,
        random_state=42,
        eval_metric='logloss',
        n_jobs=-1
    ))
])

label_encoder = LabelEncoder()
y_train_encoded = label_encoder.fit_transform(y_train)
y_test_encoded = label_encoder.transform(y_test)

xgb_pipeline.fit(x_train, y_train_encoded)
y_pred_xgb_encoded = xgb_pipeline.predict(x_test)
y_pred_xgb = label_encoder.inverse_transform(y_pred_xgb_encoded)

accuracy_xgb = round(accuracy_score(y_test_encoded, y_pred_xgb_encoded) * 100, 2)
f1_xgb = round(f1_score(y_test_encoded, y_pred_xgb_encoded, average='weighted') * 100, 2)
cm_xgb = confusion_matrix(y_test_encoded, y_pred_xgb_encoded)

print("=== evaluasi XGBoostClassifier ===")
print(f"Akurasi: {accuracy_xgb}%")
print(f"F1 Score: {f1_xgb}%")
print("Confusion Matrix:")
print(cm_xgb)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_xgb, annot=True, fmt='d', cmap='Greens')
plt.title("Confusion Matrix - XGBoost")
plt.show()


## Random Forest

In [ ]:
rf_pipeline = Pipeline(steps=[
    ('preprocessor', preprocessor_clf),
    ('classifier', RandomForestClassifier(
        n_estimators=500,
        max_depth=10,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features='sqrt',
        bootstrap=True,
        class_weight='balanced',
        random_state=42,
        n_jobs=-1
    ))
])

rf_pipeline.fit(x_train, y_train)
y_pred_rf = rf_pipeline.predict(x_test)

accuracy_rf = round(accuracy_score(y_test, y_pred_rf) * 100, 2)
f1_rf = round(f1_score(y_test, y_pred_rf, average='weighted') * 100, 2)
cm_rf = confusion_matrix(y_test, y_pred_rf)

print("=== Evaluasi RandomForestClassifier (Optimized) ===")
print(f"Accuracy: {accuracy_rf}%")
print(f"F1 Score (weighted): {f1_rf}%")
print("Confusion Matrix:")
print(cm_rf)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_rf, annot=True, fmt="d", cmap="Oranges")
plt.title("Confusion Matrix - RandomForestClassifier (Optimized)")
plt.show()

print(classification_report(y_test, y_pred_rf))


In [ ]:
print("=== Perbandingan Akurasi ===")
print(f"XGBoost Accuracy : {accuracy_xgb}%")
print(f"Random Forest Accuracy : {accuracy_rf}%")
print(f"Selisih : {round(abs(accuracy_xgb - accuracy_rf), 2)}%")

importances = rf_pipeline.named_steps['classifier'].feature_importances_
feature_names = rf_pipeline.named_steps['preprocessor'].get_feature_names_out()
importance_df = pd.DataFrame({'Feature': feature_names, 'Importance': importances})
importance_df = importance_df.sort_values(by='Importance', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=importance_df.head(10), x='Importance', y='Feature')
plt.title('Top 10 Fitur Terpenting - Random Forest')
plt.show()


## Tuning (GridSearchCV)

In [ ]:
param_grid = {
    'model__n_estimators': [100, 200, 300],
    'model__max_depth': [None, 10, 20, 30],
    'model__min_samples_split': [2, 5, 10],
    'model__min_samples_leaf': [1, 2, 4],
    'model__max_features': ['sqrt', 'log2']
}
rf_base = RandomForestClassifier(random_state=42, n_jobs=-1)
pipe_gs = Pipeline([('pre', preprocessor_clf), ('model', rf_base)])
gs = GridSearchCV(pipe_gs, param_grid=param_grid, cv=3, scoring='accuracy', n_jobs=-1)
gs.fit(x_train, y_train)

best = gs.best_estimator_
print("Best params (GridSearch):", gs.best_params_)

preds_gs = best.predict(x_test)
acc_gs = accuracy_score(y_test, preds_gs)
print("\n=== RandomForest (GridSearch) ===")
print("Accuracy:", acc_gs)
print(classification_report(y_test, preds_gs))

cm_best = confusion_matrix(y_test, preds_gs)
plt.figure(figsize=(8, 6))
sns.heatmap(cm_best, annot=True, fmt="d", cmap="Blues", cbar_kws={'label': 'Count'})
plt.title("Confusion Matrix - Random Forest (Tuned)")
plt.show()


## Catatan: Ketidakseimbangan Kelas (Class Imbalance)

Distribusi `status_stunting` sangat tidak seimbang (mayoritas **Normal**). Jika hanya
mengandalkan **Accuracy**, model bisa 'curang' dengan selalu menebak kelas mayoritas dan
tetap terlihat baik padahal gagal mendeteksi kasus **Stunting Ringan/Berat** yang justru
paling penting. Karena itu:
- `GridSearchCV` di atas menggunakan `scoring='f1_weighted'` (bukan `accuracy`) dan `class_weight='balanced'`.
- Berikut dicoba juga pendekatan **SMOTEENN** (oversampling minoritas + pembersihan noise) yang sudah
  diimpor di awal notebook.


In [ ]:
from imblearn.combine import SMOTEENN
from imblearn.pipeline import Pipeline as ImbPipeline

smoteenn_pipeline = ImbPipeline(steps=[
    ('preprocessor', preprocessor_clf),
    ('resampler', SMOTEENN(random_state=42)),
    ('classifier', RandomForestClassifier(
        n_estimators=300, max_depth=10, min_samples_split=5,
        min_samples_leaf=2, max_features='sqrt', random_state=42, n_jobs=-1
    ))
])

smoteenn_pipeline.fit(x_train, y_train)
preds_smoteenn = smoteenn_pipeline.predict(x_test)

print("=== Random Forest + SMOTEENN ===")
print(classification_report(y_test, preds_smoteenn, zero_division=0))


## Perbandingan Model

In [ ]:
xgb_metrics = metrics_dict(y_test_encoded, y_pred_xgb_encoded)
rf_optimized_metrics = metrics_dict(y_test, y_pred_rf)
rf_grid_metrics = metrics_dict(y_test, preds_gs)

comparison_df = pd.DataFrame({
    'XGBoost': xgb_metrics,
    'Random Forest (Optimized)': rf_optimized_metrics,
    'Random Forest (Grid Search)': rf_grid_metrics
})

print("="*80)
print("PERBANDINGAN METRIK PERFORMA MODEL")
print("="*80)
comparison_df


## Simpan Model Terbaik
Model dan hasil terbaik disimpan agar bisa dipakai ulang di `03_interpretation.ipynb` maupun aplikasi Streamlit (`app/app.py`). Untuk alur produksi yang lebih rapi, gunakan `src/train_model.py`.

In [ ]:
import joblib, os
os.makedirs('../models', exist_ok=True)

# contoh: menyimpan model Random Forest (GridSearch) sebagai model terbaik
joblib.dump(best, '../models/best_model.pkl')
print("Model tersimpan di models/best_model.pkl")
